# NTK kernel regression on QM9 Molecules

This notebook compares a per-atom MLP against an SO(3)-invariant CNN using
NTK kernel regression on the QM9 dataset.

The task is to compute the internal energy of each molecule (`U0_atomization`) given the positions and types of each atom. For each atom, there is an independent NN that represents that atom's environment by a spherical signal. The environment that is formed by e.g. neighboring Carbon atoms is mapped to a different channel than the one by e.g. Oxygen.
The spherical signals are discretized on a Driscoll-Healy grid defined by its bandwidth (the resolution parameter). The outputs of the individual atom's networks are then summed to a molecular vector representation. For this invariant task we compare a simple MLP-based approach vs an $\mathrm{SO}(3)$-invariant model.

## Setup

In [1]:
import jax
import jax.numpy as jnp
import neural_tangents as nt
from neural_tangents import stax
import numpy as np
from math import pi

from equivariant_ntk.layers import S2ConvSO3, SO3Pool
import dataset
import utils

jax.config.update('jax_enable_x64', True)

SEED       = 73784362
N_TRAIN    = 10
N_TEST     = 20
BANDLIMIT  = 6       # controls sphere grid resolution; Reduce for OOM issues.
POWERS     = [2, 6]  # r^-p decay channels. For each atom and each element in the dataset, we have one channel per power.
DIAG_REG   = 5e-5
BATCH_SIZE = 5       # reduce if memory is tight

TARGET    = 'U0_atomization'
MAX_ATOMS = dataset.qm9_meta['max_num_atoms']

DATA_DIR  = './data'  # override if the dataset lives elsewhere
dataset.data_dir = DATA_DIR

print(f'JAX devices: {jax.devices()}')

JAX devices: [CpuDevice(id=0)]


### Preparation: Loading Data

In [2]:
data_source = dataset.create_data_source(dataset.name, dataset.data_dir, None)

x_train, y_train = dataset.load_sphere_data(
    [TARGET], data_source['train'], True,
    BANDLIMIT, dataset.qm9_meta['atom_types'], POWERS,
    seed=SEED, max_samples=N_TRAIN,
)
x_test, y_test = dataset.load_sphere_data(
    [TARGET], data_source['test'], True,
    BANDLIMIT, dataset.qm9_meta['atom_types'], POWERS,
    seed=SEED, max_samples=N_TEST,
)

# Normalise targets by pre-computed train-set statistics
mean, std = dataset.qm9_meta['stats'][TARGET]
y_train_norm = (y_train[TARGET] - mean) / std
y_test_vals  = y_test[TARGET]           # raw Hartree values for evaluation

print(f'Train: {x_train.shape}  Test: {x_test.shape}')
print(f'Target mean={mean:.4f} Ha  std={std:.4f} Ha')

Train: (10, 29, 12, 11, 10)  Test: (20, 29, 12, 11, 10)
Target mean=-2.7983 Ha  std=0.3791 Ha


## Spherical Input Signal Visualisation

The cell below renders one channel for atom 0 of the first training molecule
at two resolutions: the model bandlimit and a
higher-resolution version for better visualisation only.

In [3]:
positions, charges, _ = dataset.load_dataset(
    [TARGET], data_source['train'], shuffle=True, seed=SEED, max_samples=1
)

VIZ_BANDLIMIT = 32   # higher resolution for visualisation only


def make_sphere_plot_data(bandlimit):
    sph = dataset.create_spherical_potentials(
        positions[0], charges[0], bandlimit,
        dataset.qm9_meta['atom_types'], POWERS,
    )
    field    = np.array(sph[0, :, :, 0])
    sph_vecs = np.array(utils.create_sphere_vecs(bandlimit))
    # close the azimuthal slice
    sph_vecs = np.concatenate([sph_vecs, sph_vecs[:, :1, :]], axis=1)
    field    = np.concatenate([field,    field[:,   :1   ]], axis=1)
    return sph_vecs, field


import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'surface'}, {'type': 'surface'}]],
    subplot_titles=[f'L = {BANDLIMIT} (model resolution)',
                    f'L = {VIZ_BANDLIMIT} (visualisation)'],
)
for col, bw in enumerate([BANDLIMIT, VIZ_BANDLIMIT], start=1):
    sph_vecs, field = make_sphere_plot_data(bw)
    xs, ys, zs = sph_vecs[..., 0], sph_vecs[..., 1], sph_vecs[..., 2]
    fig.add_trace(
        go.Surface(x=xs, y=ys, z=zs, surfacecolor=field,
                   colorscale='plasma', showscale=False),
        row=1, col=col,
    )
fig.update_layout(
    title='Spherical potential — atom 0, channel 0',
    scene=dict(aspectmode='cube'),
    scene2=dict(aspectmode='cube'),
    margin=dict(l=0, r=0, t=60, b=0),
    height=500,
)
fig.show()

## Models

Both models process atoms independently in parallel, sum the per-atom outputs,
and apply a linear head.  Channel widths are irrelevant in the infinite-width
NTK parameterisation but need to be provided.

In [4]:
from equivariant_ntk.layers import SO3ConvSO3

# Bandlimit progression for SO3CNN: S² input → SO(3) hidden → SO(3) before pool
_bws = [BANDLIMIT, BANDLIMIT // 2, BANDLIMIT // 2]

# Precompute spherical / Wigner kernels once;
precompute_sph, precompute_wig = utils.make_precompute(_bws)


def make_mlp(n_layers=2):
    """Per-atom MLP baseline (equivariant only to permutations of atoms)."""
    mlp  = stax.serial(stax.Dense(512), stax.Relu())
    single_atom = stax.serial(
        stax.Flatten(),
        stax.Dense(512),
        stax.Relu(),
        stax.repeat(mlp, n_layers - 2) if n_layers > 2 else stax.Identity(),
        stax.Dense(10),
    )
    return stax.serial(
        dataset.split_atom_features(MAX_ATOMS),
        stax.parallel(*([single_atom] * MAX_ATOMS)),
        stax.FanInSum(),
        stax.Dense(1),
    )


def make_so3cnn():
    """SO(3)-equivariant CNN: S²→SO(3) lifting, one group conv, pool, sum over atoms."""
    single_atom = stax.serial(
        S2ConvSO3(1, pi, (_bws[0], _bws[1]), precompute_sph, precompute_wig),
        stax.Erf(),
        SO3ConvSO3(1, pi, (_bws[1], _bws[2]), precompute_wig),
        stax.Erf(),
        SO3Pool(),
    )
    return stax.serial(
        dataset.split_atom_features(MAX_ATOMS),
        stax.parallel(*([single_atom] * MAX_ATOMS)),
        stax.FanInSum(),
        stax.Dense(1),
    )

## Kernel Regression

We use `nt.predict.gradient_descent_mse_ensemble` which computes the analytic
prediction of gradient-flow training at $t \to \infty$. A small diagonal regulariser `DIAG_REG` stabilises the kernel inversion for larger training sets.

In [5]:
def run_kernel_prediction(kernel_fn, x_train, y_train_norm, x_test):
    """Infinite-width NTK kernel regression; returns predictions in normalised space."""
    kf = nt.batch(kernel_fn, batch_size=BATCH_SIZE)
    predict_fn = nt.predict.gradient_descent_mse_ensemble(
        kf, x_train, y_train_norm, diag_reg=DIAG_REG,
    )
    return predict_fn(x_test=x_test, get='ntk', compute_cov=False)


def evaluate(preds_norm, y_true_Ha):
    """Denormalise predictions and compute MAE / RMSE in meV."""
    preds_meV = (np.array(preds_norm) * std + mean) * dataset.Ha_to_meV
    true_meV  = np.array(y_true_Ha)  * dataset.Ha_to_meV
    mae  = float(np.mean(np.abs(preds_meV - true_meV)))
    rmse = float(np.sqrt(np.mean((preds_meV - true_meV) ** 2)))
    return mae, rmse

### MLP

In [6]:
print('Running MLP kernel regression ...')
_, _, mlp_kernel_fn = make_mlp()
mlp_preds = run_kernel_prediction(mlp_kernel_fn, x_train, y_train_norm, x_test)
mlp_mae, mlp_rmse = evaluate(mlp_preds, y_test_vals)
print(f'MLP  MAE={mlp_mae:.1f} meV   RMSE={mlp_rmse:.1f} meV')

Running MLP kernel regression ...
MLP  MAE=5951.8 meV   RMSE=7164.4 meV


### SO3CNN

In [7]:
print('Running SO3CNN kernel regression ...')
_, _, so3_kernel_fn = make_so3cnn()
so3_preds = run_kernel_prediction(so3_kernel_fn, x_train, y_train_norm, x_test)
so3_mae, so3_rmse = evaluate(so3_preds, y_test_vals)
print(f'SO3CNN  MAE={so3_mae:.1f} meV   RMSE={so3_rmse:.1f} meV')

Running SO3CNN kernel regression ...
SO3CNN  MAE=3340.9 meV   RMSE=4510.9 meV


### Results

In [8]:
print(f'Infinite-width NTK predictions  ({N_TRAIN} train / {N_TEST} test, bandlimit={BANDLIMIT})')
print(f'{"":-<48}')
print(f'{"Model":10s}  {"MAE (meV)":>12s}  {"RMSE (meV)":>12s}')
print(f'{"":-<48}')
print(f'{"MLP":10s}  {mlp_mae:>12.1f}  {mlp_rmse:>12.1f}')
print(f'{"SO3CNN":10s}  {so3_mae:>12.1f}  {so3_rmse:>12.1f}')

Infinite-width NTK predictions  (10 train / 20 test, bandlimit=6)
------------------------------------------------
Model          MAE (meV)    RMSE (meV)
------------------------------------------------
MLP               5951.8        7164.4
SO3CNN            3340.9        4510.9
